In [1]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 1.3 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/8.4 MB 1.3 MB/s eta 0:00:06
   ----- ---------------------------------- 1.0/8.4 MB 1.2 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.4 MB 1.1 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.4 MB 1.1 MB/s eta 0:00:06
   -------- ------------------------------- 1.8/8.4 MB 1.2 MB/s eta 0:00:06
   ---------- ----------------------------- 2.1/8.4 MB 1.2 MB/s eta 0:00:06
   ---------- ----------------------------- 2.1/8.4 MB 1.2 MB/s eta 0:00:06
   ----------- ---------------------------- 2.4/8.4 MB 1.1 MB/s eta 0:00:06
   ------------ --------------------------- 2.6/8.4 MB 1.2 MB/s eta 0:00:05
   ------------- ----------------


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ============================================================
# AI USE CASE 1: LATE-DELIVERY RISK SCORING
# Predicts, at order time, whether an item will be delivered late.
# Same business rules as SQL / Python EDA / Power BI:
#   window 2015-01-01 to 2017-09-30, canceled shipments excluded.
# Outputs: late_risk_scores.csv (for Power BI) and model_metrics.csv
# ============================================================
# pip install pandas numpy scikit-learn openpyxl

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict

WINDOW_START, WINDOW_END = pd.Timestamp("2015-01-01"), pd.Timestamp("2017-09-30")
TRAIN_END = pd.Timestamp("2017-01-01")      # train on 2015-2016, test on 2017
FILE_PATH = Path("DataCo.xlsx")

# ------------------------------------------------------------
# 1. LOAD AND CLEAN (same steps as supply_chain_eda_v2.py)
# ------------------------------------------------------------
df = pd.read_excel(FILE_PATH, engine="openpyxl")
df.columns = (df.columns.astype(str).str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))


def parse_mixed_date(value):
    """Text cells are M/D/YYYY; real Excel dates have day and month swapped."""
    if isinstance(value, str):
        return pd.to_datetime(value.strip(), format="%m/%d/%Y %H:%M")
    ts = pd.Timestamp(value)
    return pd.Timestamp(year=ts.year, month=ts.day, day=ts.month,
                        hour=ts.hour, minute=ts.minute)


df["order_ts"] = df["order_date_dateorders"].map(parse_mixed_date)
df["ship_ts"] = df["shipping_date_dateorders"].map(parse_mixed_date)
gap = (df["ship_ts"].dt.normalize() - df["order_ts"].dt.normalize()).dt.days
assert (gap == df["days_for_shipping_real"]).all(), "Date parsing is wrong"

d = df[df["order_ts"].dt.normalize().between(WINDOW_START, WINDOW_END)
       & (df["delivery_status"] != "Shipping canceled")].copy()
d["late"] = (d["late_delivery_risk"] == 1).astype(int)
print(f"Items scored: {len(d):,} | late rate: {d['late'].mean():.2%}")

# ------------------------------------------------------------
# 2. FEATURES - ONLY WHAT IS KNOWN AT ORDER TIME
# NOT used (would leak the answer): days_for_shipping_real, delivery_status,
# shipping_date, late_delivery_risk. Tested and dropped because they added
# no signal: market, region, category, segment, payment type, quantity,
# discount, price, month, weekday.
# ------------------------------------------------------------
d["order_hour"] = d["order_ts"].dt.hour
FEATURES = ["shipping_mode", "days_for_shipment_scheduled", "order_hour"]
X = d[FEATURES].copy()
X["shipping_mode"] = X["shipping_mode"].astype("category")
y = d["late"]


def make_model():
    return HistGradientBoostingClassifier(
        max_iter=150, learning_rate=0.1, max_depth=4,
        categorical_features="from_dtype", random_state=42)


# ------------------------------------------------------------
# 3. BASELINES (a model must beat these to be worth using)
# ------------------------------------------------------------
train = (d["order_ts"] < TRAIN_END).values
test = ~train

def rule_predict(frame):
    """Plain-language rule, no machine learning."""
    mode, hour = frame["shipping_mode"], frame["order_hour"]
    return np.select(
        [mode == "First Class", mode == "Second Class", mode == "Same Day"],
        [1, 1, (hour >= 12).astype(int)], default=0)


def metrics(name, y_true, pred, score=None):
    return {"method": name,
            "accuracy": accuracy_score(y_true, pred),
            "precision_late": precision_score(y_true, pred),
            "recall_late": recall_score(y_true, pred),
            "auc": roc_auc_score(y_true, score if score is not None else pred)}

rows = []
rows.append(metrics("Always predict late", y[test], np.ones(test.sum(), dtype=int)))

mode_rate = d[train].groupby("shipping_mode")["late"].mean()
mode_score = d[test]["shipping_mode"].map(mode_rate)
rows.append(metrics("Shipping mode only", y[test], (mode_score > 0.5).astype(int), mode_score))

rows.append(metrics("Rule: mode + Same Day noon cutoff", y[test], rule_predict(d[test])))

model = make_model().fit(X[train], y[train])
prob_test = model.predict_proba(X[test])[:, 1]
rows.append(metrics("Gradient boosting (3 features)", y[test], (prob_test >= 0.5).astype(int), prob_test))

report = pd.DataFrame(rows)
print("\n--- TEST RESULTS: train 2015-2016, test Jan-Sep 2017 ---")
print(report.round(4).to_string(index=False))
report.to_csv("model_metrics.csv", index=False)

# ------------------------------------------------------------
# 4. WHAT THE DATA ACTUALLY SAYS (so the result can be explained)
# ------------------------------------------------------------
d["hour_band"] = np.where(d["order_hour"] < 12, "before noon", "noon or later")
print("\n--- LATE RATE BY SHIPPING MODE AND ORDER TIME ---")
print((d.pivot_table(index="shipping_mode", columns="hour_band",
                     values="late", aggfunc="mean") * 100).round(1))

# ------------------------------------------------------------
# 5. SCORE EVERY ITEM WITH OUT-OF-FOLD PREDICTIONS
# Each item is scored by a model that never saw its order, so the
# scores are honest. Folds are grouped by order_id.
# ------------------------------------------------------------
oof = cross_val_predict(make_model(), X, y, groups=d["order_id"],
                        cv=GroupKFold(n_splits=5), method="predict_proba")[:, 1]
d["late_probability"] = oof.round(4)
d["risk_band"] = pd.cut(d["late_probability"], [-0.01, 0.30, 0.70, 1.01],
                        labels=["Low", "Medium", "High"]).astype(str)

print("\n--- RISK BANDS (out-of-fold) ---")
band = d.groupby("risk_band").agg(items=("late", "size"), actual_late_rate=("late", "mean"))
band["actual_late_rate"] = (band["actual_late_rate"] * 100).round(1)
print(band)

out = d[["order_item_id", "order_id", "shipping_mode", "order_hour",
         "late_probability", "risk_band", "late"]].rename(columns={"late": "actual_late"})
out.to_csv("late_risk_scores.csv", index=False)
print(f"\nSaved late_risk_scores.csv ({len(out):,} rows) and model_metrics.csv")

# ------------------------------------------------------------
# INTERPRETATION
# - Only shipping mode and order hour carry signal. The hour effect exists
#   only for Same Day: orders before noon are never late, orders at noon or
#   later are always late.
# - Second Class and Standard Class outcomes are not explained by any
#   available field, so no model can do much better than 72% on this data.
# - Delivery fields look simulated: use as a method demonstration, and
#   validate against real carrier data before operational use.
# ------------------------------------------------------------

Items scored: 164,572 | late rate: 57.28%

--- TEST RESULTS: train 2015-2016, test Jan-Sep 2017 ---
                           method  accuracy  precision_late  recall_late    auc
              Always predict late    0.5695          0.5695        1.000 0.5000
               Shipping mode only    0.6934          0.8819        0.533 0.7404
Rule: mode + Same Day noon cutoff    0.7219          0.8910        0.583 0.7443
   Gradient boosting (3 features)    0.7219          0.8910        0.583 0.7737

--- LATE RATE BY SHIPPING MODE AND ORDER TIME ---
hour_band       before noon  noon or later
shipping_mode                             
First Class           100.0          100.0
Same Day                0.0          100.0
Second Class           79.8           79.8
Standard Class         39.6           40.0

--- RISK BANDS (out-of-fold) ---
           items  actual_late_rate
risk_band                         
High       61681              89.5
Low         4628               0.0
Medium     98263 